In [ ]:
# 1. Colab ortamında çakışma yaşandığından dolayı gereksiz kütüphaneler silinmiştir.
!pip uninstall -y tensorflow tensorboard

# 2. Sadece ihtiyacımız olan araçları, çakışma olmadan (ve doğru sürümle) kuruyoruz
!pip install torch torchvision matplotlib kaggle mediapipe==0.10.14 opencv-python

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tensorboard 2.20.0
Uninstalling tensorboard-2.20.0:
  Successfully uninstalled tensorboard-2.20.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 27.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 0.10.35
    Uninstalling mediapipe-0.10.35:
      Successfully uninstalled mediapipe-0.10.35
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.2

In [ ]:
from google.colab import drive
import os

# 1. Google Drive'ı Colab'a bağlıyoruz
drive.mount('/content/drive')

# 2. Drive'ının içinde modellerimiz için güvenli bir klasör oluşturuyoruz
drive_klasoru = '/content/drive/MyDrive/Meme_GAN_Modellerim'
os.makedirs(drive_klasoru, exist_ok=True)

print(f"Klasör hazır: {drive_klasoru}")

In [ ]:
import os
import glob
from google.colab import userdata

# 1. Colab Secrets ile Kimlik Doğrulama
os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')
print("Kaggle kimlik doğrulaması başarılı!")

# 2. Veri setini indir
!kaggle datasets download -d yashwantk23cse/deep-fashion

# 3. Klasörü oluştur ve zip'i sessizce, üzerine yazarak çıkart
!mkdir -p dataset
!unzip -q -o deep-fashion.zip -d dataset/

# 4. (Disk Temizliği) Ön işlemeye yer açmak için zip'i sil
!rm deep-fashion.zip
print("Disk temizliği yapıldı (Zip dosyası silindi).")

# 5. Sistem Doğrulaması
image_paths = glob.glob("dataset/**/*.jpg", recursive=True)
print(f"Harika! Veri seti çıkartıldı ve toplam {len(image_paths)} resim Ön İşleme için hazır.")

Kaggle kimlik doğrulaması başarılı!
Dataset URL: https://www.kaggle.com/datasets/yashwantk23cse/deep-fashion
License(s): CC-BY-NC-SA-4.0
100% 14.7G/14.7G [07:13<00:00, 36.5MB/s]

Disk temizliği yapıldı (Zip dosyası silindi).
Harika! Veri seti çıkartıldı ve toplam 286743 resim Ön İşleme için hazır.


In [ ]:
import os
import glob
import cv2
import mediapipe as mp
import numpy as np

# model silme

In [ ]:
import os

# Silinecek eski modellerin yolları
eski_model = "/content/drive/MyDrive/Meme_GAN_Modellerim/wgan_model.pth"
eski_best_model = "/content/drive/MyDrive/Meme_GAN_Modellerim/wgan_best_model.pth"

silinecekler = [eski_model, eski_best_model]

print("Temizlik başlatılıyor...\n")

for dosya in silinecekler:
    if os.path.exists(dosya):
        os.remove(dosya)
        print(f"BAŞARIYLA SİLİNDİ: {os.path.basename(dosya)}")
    else:
        print(f"Zaten silinmiş veya yok: {os.path.basename(dosya)}")

print("\nDrive temizliği tamamlandı! Artık V2 mimarisiyle o kusursuz eğitime başlamaya hazırsın.")

# önişleme (updated with kategories)



In [ ]:
import os
import glob
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm
import warnings


warnings.filterwarnings("ignore")

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

image_paths = glob.glob("dataset/**/*.jpg", recursive=True)[:5000]
print(f"Toplam {len(image_paths)} resim için eşleştirilmiş iskelet çıkarılıyor...\n")

successful_poses = 0

for path in tqdm(image_paths, desc="İskeletler Çıkarılıyor"):
    pose_path = path.replace("dataset", "dataset_poses")
    os.makedirs(os.path.dirname(pose_path), exist_ok=True)

    if not os.path.exists(pose_path):
        with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
            image = cv2.imread(path)
            if image is not None:
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                results = pose.process(image_rgb)
                blank_image = np.zeros(image.shape, dtype=np.uint8)
                if results.pose_landmarks:
                    mp_drawing.draw_landmarks(
                        blank_image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                        mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                        mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                    )
                    cv2.imwrite(pose_path, blank_image)
                    successful_poses += 1
    else:
        # Eğer iskelet zaten önceden çıkarılmışsa (önceki denemeden kalanlar) atla
        successful_poses += 1

print(f"\nÖn İşleme bitti! Toplam {successful_poses} iskelet başarıyla çıkarıldı.")

Toplam 5000 resim için eşleştirilmiş iskelet çıkarılıyor...



İskeletler Çıkarılıyor: 100%|██████████| 5000/5000 [14:24<00:00,  5.78it/s]


Ön İşleme bitti! Toplam 3334 iskelet başarıyla çıkarıldı.


# Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import random
import glob
import os
import time
from torch import autograd

# ==========================================
# 1. KOPYA ÇEKMEYİ ENGELLEYEN VERİ SETİ (TURBO MODU)
# ==========================================
class PairedPoseDataset(Dataset):
    def __init__(self, target_paths):
        self.target_paths = target_paths
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        self.anti_cheat_transform = transforms.RandomHorizontalFlip(p=1.0)

        print("Klasör haritası RAM'e yükleniyor...")
        self.dir_cache = {}
        for p in self.target_paths:
            d = os.path.dirname(p)
            if d not in self.dir_cache:
                self.dir_cache[d] = [img for img in glob.glob(os.path.join(d, "*.jpg"))]

    def __len__(self): return len(self.target_paths)

    def __getitem__(self, idx):
        try:
            target_path = self.target_paths[idx]
            pose_path = target_path.replace("dataset", "dataset_poses")

            target_dir = os.path.dirname(target_path)
            other_images = [img for img in self.dir_cache[target_dir] if img != target_path]

            if len(other_images) > 0:
                identity_img = Image.open(random.choice(other_images)).convert('RGB')
            else:
                identity_img = self.anti_cheat_transform(Image.open(target_path).convert('RGB'))

            identity_tensor = self.transform(identity_img)
            pose_tensor = self.transform(Image.open(pose_path).convert('RGB'))
            target_tensor = self.transform(Image.open(target_path).convert('RGB'))

            return identity_tensor, pose_tensor, target_tensor
        except:
            return torch.zeros(3, 256, 256), torch.zeros(3, 256, 256), torch.zeros(3, 256, 256)

valid_targets = [p for p in glob.glob("dataset/**/*.jpg", recursive=True) if os.path.exists(p.replace("dataset", "dataset_poses"))]

if len(valid_targets) == 0:
    print("HATA: İskelet bulunamadı! Hücre 3'ü çalıştırın.")
else:
    random.shuffle(valid_targets)
    split_idx = int(len(valid_targets) * 0.9)
    train_loader = DataLoader(PairedPoseDataset(valid_targets[:split_idx]), batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(PairedPoseDataset(valid_targets[split_idx:]), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 2. WGAN MODELLERİ & OPTİMİZASYON
# ==========================================
class UNetGenerator(nn.Module):
    def __init__(self):
        super(UNetGenerator, self).__init__()
        self.d1 = self.cb(6, 64, False); self.d2 = self.cb(64, 128); self.d3 = self.cb(128, 256); self.d4 = self.cb(256, 512)
        self.u1 = self.db(512, 256); self.u2 = self.db(256, 128); self.u3 = self.db(128, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
    def cb(self, ic, oc, n=True):
        l = [nn.Conv2d(ic, oc, 4, 2, 1)]
        if n: l.append(nn.InstanceNorm2d(oc))
        l.append(nn.LeakyReLU(0.2, True))
        return nn.Sequential(*l)
    def db(self, ic, oc):
        return nn.Sequential(nn.ConvTranspose2d(ic, oc, 4, 2, 1), nn.InstanceNorm2d(oc), nn.ReLU(True))
    def forward(self, i, p): return self.final(self.u3(self.u2(self.u1(self.d4(self.d3(self.d2(self.d1(torch.cat([i, p], 1)))))))))

class WGANCritic(nn.Module):
    def __init__(self):
        super(WGANCritic, self).__init__()
        def cb(ic, oc, n=True):
            l = [nn.Conv2d(ic, oc, 4, 2, 1)]
            if n: l.append(nn.InstanceNorm2d(oc))
            l.append(nn.LeakyReLU(0.2, True))
            return nn.Sequential(*l)
        self.m = nn.Sequential(cb(9, 64, False), cb(64, 128), cb(128, 256), nn.Conv2d(256, 1, 4, 1, 1))
    def forward(self, i, p, img): return self.m(torch.cat([i, p, img], 1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = UNetGenerator().to(device)
critic = WGANCritic().to(device)

# --- YENİ HASSAS ÖĞRENME ORANI ---
yeni_lr = 0.00005
optimizer_G = optim.Adam(generator.parameters(), lr=yeni_lr, betas=(0.0, 0.9))
optimizer_C = optim.Adam(critic.parameters(), lr=yeni_lr, betas=(0.0, 0.9))
lambda_gp = 10

def compute_gradient_penalty(critic, real_samples, fake_samples, identity, pose):
    alpha = torch.rand((real_samples.size(0), 1, 1, 1)).to(device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = critic(identity, pose, interpolates)
    fake = torch.ones((real_samples.size(0), 1, 31, 31)).to(device)
    gradients = autograd.grad(outputs=d_interpolates, inputs=interpolates, grad_outputs=fake, create_graph=True, retain_graph=True, only_inputs=True)[0]
    return ((gradients.view(gradients.size(0), -1).norm(2, dim=1) - 1) ** 2).mean()

# ==========================================
# 3. EĞİTİM DÖNGÜSÜ & YENİ PATIENCE
# ==========================================
epochs = 200

drive_folder = "/content/drive/MyDrive/Meme_GAN_Modellerim"
os.makedirs(drive_folder, exist_ok=True)
checkpoint_path_latest = os.path.join(drive_folder, "wgan_model_v2.pth")
checkpoint_path_best = os.path.join(drive_folder, "wgan_best_model_v2.pth")

start_epoch = 0
patience_limit = 20 # Limiti tekrar normale çektik
patience_counter = 0
best_val_loss = float('inf')

if len(valid_targets) > 0:
    if os.path.exists(checkpoint_path_latest):
        print(f"\nDrive'dan v2 Checkpoint yükleniyor...")
        cp = torch.load(checkpoint_path_latest)
        generator.load_state_dict(cp['g_state']); critic.load_state_dict(cp['c_state'])
        optimizer_G.load_state_dict(cp['o_g']); optimizer_C.load_state_dict(cp['o_c'])
        start_epoch = cp['epoch'] + 1
        if 'best_val_loss' in cp: best_val_loss = cp['best_val_loss']
        if 'patience_counter' in cp: patience_counter = cp['patience_counter']

        # KRİTİK MÜDAHALE: Drive'dan gelen eski hızlı (0.0001) hızı ezip, yavaş (0.00005) hızı zorla!
        for param_group in optimizer_G.param_groups: param_group['lr'] = yeni_lr
        for param_group in optimizer_C.param_groups: param_group['lr'] = yeni_lr

    print(f"\n---EĞİTİM BAŞLIYOR | {device} ---")
    for epoch in range(start_epoch, epochs):
        start_time = time.time()
        generator.train(); critic.train()
        t_g_loss, t_c_loss, v_batches = 0, 0, 0

        for identity, pose, target in train_loader:
            if torch.sum(identity) == 0: continue
            identity, pose, target = identity.to(device), pose.to(device), target.to(device)

            # Critic Eğitimi
            optimizer_C.zero_grad()
            gen_img = generator(identity, pose).detach()
            loss_C = -torch.mean(critic(identity, pose, target)) + torch.mean(critic(identity, pose, gen_img))
            gp = compute_gradient_penalty(critic, target, gen_img, identity, pose)
            total_loss_C = loss_C + lambda_gp * gp
            total_loss_C.backward()
            optimizer_C.step()
            t_c_loss += total_loss_C.item()

            # Generator Eğitimi
            optimizer_G.zero_grad()
            gen_img = generator(identity, pose)
            loss_G = -torch.mean(critic(identity, pose, gen_img)) + (100 * nn.L1Loss()(gen_img, target))
            loss_G.backward()
            optimizer_G.step()
            t_g_loss += loss_G.item()
            v_batches += 1

        # Validation (Doğrulama)
        generator.eval(); val_l1 = 0; val_b = 0
        with torch.no_grad():
            for vi, vp, vt in val_loader:
                if torch.sum(vi) == 0: continue
                vg = generator(vi.to(device), vp.to(device))
                val_l1 += nn.L1Loss()(vg, vt.to(device)).item()
                val_b += 1

        current_val_loss = val_l1 / val_b if val_b > 0 else 0

        print(f"[Epoch {epoch+1}/{epochs}] [Süre: {time.time()-start_time:.1f}s] | "
              f"Critic L: {t_c_loss/v_batches:.8f} | Gen L: {t_g_loss/v_batches:.8f} | Val L1: {current_val_loss:.8f}")

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            patience_counter = 0
            torch.save({'g_state': generator.state_dict()}, checkpoint_path_best)
            print(f"Yeni En İyi Model! -> 'wgan_best_model_v2.pth' güncellendi.")
        else:
            patience_counter += 1
            print(f"İyileşme yok. Sabır Sayacı: {patience_counter}/{patience_limit}")

        if patience_counter >= patience_limit:
            print(f"\nERKEN DURDURMA TETİKLENDİ! Eğitim {patience_limit} epoch boyunca gelişmedi.")
            break

        if (epoch + 1) % 5 == 0:
            torch.save({'epoch': epoch, 'g_state': generator.state_dict(), 'c_state': critic.state_dict(),
                        'o_g': optimizer_G.state_dict(), 'o_c': optimizer_C.state_dict(),
                        'best_val_loss': best_val_loss, 'patience_counter': patience_counter}, checkpoint_path_latest)

# Test öncesi çalıştırma

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


# TEST

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from google.colab import files
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. JENERATÖR MİMARİSİ
# ==========================================
class UNetGenerator(nn.Module):
    def __init__(self):
        super(UNetGenerator, self).__init__()
        self.d1 = self.cb(6, 64, False); self.d2 = self.cb(64, 128); self.d3 = self.cb(128, 256); self.d4 = self.cb(256, 512)
        self.u1 = self.db(512, 256); self.u2 = self.db(256, 128); self.u3 = self.db(128, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
    def cb(self, ic, oc, n=True):
        l = [nn.Conv2d(ic, oc, 4, 2, 1)]
        if n: l.append(nn.InstanceNorm2d(oc))
        l.append(nn.LeakyReLU(0.2, True))
        return nn.Sequential(*l)
    def db(self, ic, oc):
        return nn.Sequential(nn.ConvTranspose2d(ic, oc, 4, 2, 1), nn.InstanceNorm2d(oc), nn.ReLU(True))
    def forward(self, i, p): return self.final(self.u3(self.u2(self.u1(self.d4(self.d3(self.d2(self.d1(torch.cat([i, p], 1)))))))))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = UNetGenerator().to(device)

# --- V2 MODELİNİ YÜKLE ---
drive_model_path = "/content/drive/MyDrive/Meme_GAN_Modellerim/wgan_best_model_v2.pth"
if os.path.exists(drive_model_path):
    print("Drive'dan model çekiliyor...\n")
    checkpoint = torch.load(drive_model_path, map_location=device)
    generator.load_state_dict(checkpoint['g_state'])
    generator.eval()
else:
    print("HATA: Model Drive'da bulunamadı! Yolun doğruluğunu kontrol edin.")

# ==========================================
# 2. İNTERAKTİF DOSYA YÜKLEME ARAYÜZÜ
# ==========================================
print("1. ADIM: Lütfen KİŞİ FOTOĞRAFINI (Kendi resminiz) seçin:")
uploaded_id = files.upload()
if not uploaded_id:
    raise ValueError("Fotoğraf yüklemesi iptal edildi!")
kendi_fotografim_yolu = list(uploaded_id.keys())[0]

print("\n2. ADIM: Lütfen MEME TEMPLATE (Duruş/İskelet için) resmini seçin:")
uploaded_pose = files.upload()
if not uploaded_pose:
    raise ValueError("Meme template yüklemesi iptal edildi!")
meme_template_yolu = list(uploaded_pose.keys())[0]

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ==========================================
# 3. İSKELET ÇIKARMA MOTORU
# ==========================================
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def extract_skeleton_from_image(image_path):
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        image = cv2.imread(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        blank_skeleton = np.zeros((256, 256, 3), dtype=np.uint8)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                blank_skeleton, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
            )
        return Image.fromarray(blank_skeleton)

# ==========================================
# 4. MEME ÜRETİM VE GÖSTERİM
# ==========================================
print("\n Meme hazırlanıyor...")

try:
    img_id = Image.open(kendi_fotografim_yolu).convert('RGB')
    t_id = transform(img_id).unsqueeze(0).to(device)

    img_skeleton = extract_skeleton_from_image(meme_template_yolu)
    t_pose = transform(img_skeleton).unsqueeze(0).to(device)

    # ÜRETİM
    with torch.no_grad():
        gen_tensor = generator(t_id, t_pose).squeeze(0).cpu()

    gen_img = transforms.ToPILImage()((gen_tensor * 0.5) + 0.5)

    # Çıktı Gösterimi
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_id)
    axes[0].set_title("Kişi (Identity)")
    axes[0].axis('off')

    axes[1].imshow(img_skeleton)
    axes[1].set_title("Meme İskeleti (Pose)")
    axes[1].axis('off')

    axes[2].imshow(gen_img)
    axes[2].set_title("Üretilen Görsel")
    axes[2].axis('off')

    plt.show()

    # Sonucu Kaydet
    drive_folder = "/content/drive/MyDrive/Meme_GAN_Modellerim"
    os.makedirs(drive_folder, exist_ok=True)
    final_save_path = f"{drive_folder}/v2_meme_{int(time.time())}.jpg"
    gen_img.save(final_save_path)
    print(f"Sonuç Drive'a kaydedildi: {final_save_path}")

except Exception as e:
    print(f"HATA OLUŞTU: {e}")

# Bu noktadan itibaren model başarılı sonuç gösterememiştir ve StableDiffusion/ControlNet mimarisine geçilmiştir.